In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

In [36]:
df = pd.read_csv("WeatherJena.csv")

In [37]:
df.head(10)

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg),rain (mm),raining (s),SWDR (W/m�),PAR (�mol/m�/s),max. PAR (�mol/m�/s),Tlog (degC),CO2 (ppm)
0,01.01.2010 00:10:00,967.56,-2.84,272.89,-3.41,95.8,4.95,4.75,0.21,3.06,4.91,1244.56,1.61,2.76,15.41,0.0,0,0.0,0.0,0.0,8.59,404.6
1,01.01.2010 00:20:00,967.45,-2.85,272.88,-3.43,95.7,4.95,4.74,0.21,3.05,4.90,1244.48,2.00,3.10,17.04,0.0,0,0.0,0.0,0.0,8.53,404.1
2,01.01.2010 00:30:00,967.45,-2.88,272.85,-3.46,95.8,4.94,4.73,0.21,3.05,4.89,1244.65,2.25,3.79,25.35,0.0,0,0.0,0.0,0.0,8.52,404.3
3,01.01.2010 00:40:00,967.34,-2.90,272.84,-3.47,95.8,4.93,4.72,0.21,3.04,4.88,1244.57,2.64,3.77,23.64,0.0,270,0.0,0.0,0.0,8.54,404.0
4,01.01.2010 00:50:00,967.29,-2.96,272.78,-3.53,95.8,4.91,4.70,0.21,3.03,4.86,1244.81,2.82,4.29,18.94,0.0,310,0.0,0.0,0.0,8.55,401.9
5,01.01.2010 01:00:00,967.18,-3.03,272.72,-3.59,95.9,4.88,4.68,0.20,3.02,4.84,1245.00,2.92,4.09,22.06,0.0,600,0.0,0.0,0.0,8.55,401.6
6,01.01.2010 01:10:00,967.14,-3.05,272.70,-3.59,96.0,4.88,4.68,0.20,3.02,4.84,1245.03,2.70,4.93,18.60,0.0,60,0.0,0.0,0.0,8.52,401.8
7,01.01.2010 01:20:00,967.10,-3.03,272.73,-3.56,96.1,4.88,4.69,0.19,3.02,4.85,1244.89,2.28,3.53,20.80,0.1,0,0.0,0.0,0.0,8.46,402.3
8,01.01.2010 01:30:00,967.09,-3.04,272.72,-3.56,96.2,4.88,4.69,0.19,3.02,4.85,1244.93,2.51,4.51,23.99,0.0,250,0.0,0.0,0.0,8.42,402.8
9,01.01.2010 01:40:00,967.13,-3.10,272.66,-3.60,96.3,4.86,4.68,0.18,3.01,4.84,1245.24,2.83,3.93,23.12,0.0,40,0.0,0.0,0.0,8.39,402.5


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 656956 entries, 0 to 656955
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Date Time             656956 non-null  object 
 1   p (mbar)              656956 non-null  float64
 2   T (degC)              656956 non-null  float64
 3   Tpot (K)              656956 non-null  float64
 4   Tdew (degC)           656956 non-null  float64
 5   rh (%)                656956 non-null  float64
 6   VPmax (mbar)          656956 non-null  float64
 7   VPact (mbar)          656956 non-null  float64
 8   VPdef (mbar)          656956 non-null  float64
 9   sh (g/kg)             656956 non-null  float64
 10  H2OC (mmol/mol)       656956 non-null  float64
 11  rho (g/m**3)          656956 non-null  float64
 12  wv (m/s)              656956 non-null  float64
 13  max. wv (m/s)         656956 non-null  float64
 14  wd (deg)              656956 non-null  float64
 15  

In [39]:
df.shape

(656956, 22)

In [40]:
feature = ["T (degC)",
           "p (mbar)",
           "rh (%)",
           "wv (m/s)",
           "max. wv (m/s)"]

In [41]:
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df = df[df["Date Time"] <= "31.12.2015 23:50:00"]
df.set_index('Date Time', inplace=True)


In [42]:
data = df[feature].resample('h').mean().dropna()

In [43]:
data.head()

,T (degC),p (mbar),rh (%),wv (m/s),max. wv (m/s)
Date Time,,,,,
2010-01-01 00:00:00,-2.886000,967.418000,95.780000,2.264000,3.542000
2010-01-01 01:00:00,-3.058333,967.113333,96.133333,2.713333,4.220000
2010-01-01 02:00:00,-3.118333,967.033333,96.283333,2.816667,3.995000
2010-01-01 03:00:00,-3.435000,967.031667,96.566667,3.541667,4.983333
2010-01-01 04:00:00,-3.731667,966.805000,96.950000,3.286667,4.553333


In [45]:
train_split = int(len(data)*0.8)

In [47]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(data.iloc[:train_split])
test_scaled = scaler.transform(data.iloc[train_split:])

In [82]:
def ceate_time_stapes(data, time_step = 24):
  x, y = [], []
  for i in range(time_step, len(data)):
    # print(i)
    x.append(data[i-time_step:i, 1:])
    y.append(data[i, 0])
  return np.array(x), np.array(y)

In [83]:
x_train, y_train = ceate_time_stapes(train_scaled,24)

In [84]:
y_train[:5]

array([0.29926014, 0.29920095, 0.29751406, 0.29600474, 0.29304528])

In [85]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input

In [87]:
model = Sequential([
    Input(shape=(24,4)),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="linear")
])

In [88]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[
        tf.keras.metrics.R2Score(name="R2")
    ]
)

In [89]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_12 (LSTM)                  │ (None, 24, 256)        │       267,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 24, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 24, 128)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 516,417 (1.97 MB)

 Trainable params: 516,417 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

In [90]:
model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=2,
    validation_split=0.2,
    verbose=1
)

Epoch 1/2
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 157s 144ms/step - R2: 0.1747 - loss: 0.0205 - val_R2: 0.0089 - val_loss: 0.0166
Epoch 2/2
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 155s 147ms/step - R2: 0.4100 - loss: 0.0147 - val_R2: 0.1864 - val_loss: 0.0136
